# 학습 방식 네 가지 — 코드와 성적표

같은 과제(`primary_action` 3분류)를 푸는 방식이 넷이다. 각각 **무엇을 학습하는지**와
**실제 코드가 어떻게 생겼는지**를 나란히 본다.

| | 방식 | 학습 대상 | 상태 |
|---|---|---|---|
| **A** | TF-IDF + 선형 분류기 | 문자·단어 n-gram 가중치 | 현행 기준선 |
| **B** | 인코더 파인튜닝 | 사전학습 트랜스포머 전체 | v4에서 측정 |
| **C** | 앙상블 (soft/hard voting) | 학습 없음 — 저장된 OOF를 묶음 | v4에서 측정 |
| **D** | 다중 헤드 (멀티태스크) | 공유 인코더 + 보조 과제 | **미구현** |

읽는 법: 점수는 전부 **fold 평균 macro F1**으로 통일한다. 통합 OOF는 같은 모델에서도
0.02쯤 높게 나오므로 섞어 읽으면 안 된다(§마지막 셀).

In [ ]:
"""성적표 — 네 방식의 실측값을 한 자리에 모은다."""
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

# v4는 파인튜닝·앙상블이 측정된 유일한 데이터셋이다(GPU가 필요해 v5 재학습이 남았다).
# v5는 A만 있다. **같은 열에 놓되 데이터셋이 다르다는 것을 표에 남긴다.**
finetune = json.loads((ROOT / "reports/current/v4/finetune_results.json").read_text("utf-8"))
singles = finetune["singles"]
ensembles = finetune["ensembles"]

board = pd.DataFrame(
    [
        ("A  TF-IDF word+char", "v4", singles["word+char TF-IDF"]["fold_mean_macro_f1"],
         singles["word+char TF-IDF"]["pooled_macro_f1"], singles["word+char TF-IDF"]["boundary_errors"]),
        ("B  파인튜닝 최고 단일 ftL42", "v4", singles["ftL42"]["fold_mean_macro_f1"],
         singles["ftL42"]["pooled_macro_f1"], singles["ftL42"]["boundary_errors"]),
        ("C  앙상블 wc+ftB7+ftL42", "v4", ensembles["wc+ftB7+ftL42"]["fold_mean_macro_f1"],
         ensembles["wc+ftB7+ftL42"]["pooled_macro_f1"],
         ensembles["wc+ftB7+ftL42"]["boundary_errors"]),
        ("D  다중 헤드", "—", float("nan"), float("nan"), float("nan")),
        ("A  TF-IDF word+char", "v5", 0.6395, float("nan"), 127.0),
    ],
    columns=["방식", "데이터셋", "fold평균 macroF1", "통합 OOF", "경계 혼동"],
)
print(board.round(3).to_string(index=False, na_rep="—"))

base, best = singles["word+char TF-IDF"], ensembles["wc+ftB7+ftL42"]
print(f"\nv4에서 A→C 이득: {best['fold_mean_macro_f1'] - base['fold_mean_macro_f1']:+.3f}")
print(f"  같은 구간 경계 혼동: {base['boundary_errors']} → {best['boundary_errors']}건")
print("  → **점수는 오르지만 경계는 그대로다.** 이득은 경계 밖에서 나온다.")

In [ ]:
"""A. TF-IDF + 선형 — 실제 코드와 1 fold 실행.

`ModelSpec`이 벡터라이저와 분류기를 한 곳에서 정한다. dataclass라서 `replace()`로
한 축만 바꿔 대조군을 만들 수 있다 — 어제 트리 판정에서 쓴 방법이다.
"""
import inspect
import re

from sklearn.metrics import f1_score

from scripts.evaluation import baselines as B
from scripts.evaluation.folds import make_lodo_folds
from scripts.labeling.label_dataset import load_label_dataset

print("― 스펙 정의 " + "―" * 40)
# docstring을 건너뛰고 필드 목록만 보여준다. 이 dataclass 한 곳이 벡터라이저와
# 분류기와 가중치를 전부 정하므로, 여기만 보면 A가 무엇을 학습하는지 알 수 있다.
spec_src = inspect.getsource(B.ModelSpec).split("def build")[0]
print(spec_src[spec_src.index("    name: str"):].rstrip())

print("\n― 분류기 선택부 " + "―" * 36)
build = inspect.getsource(B.ModelSpec.build)
print("\n".join(re.search(r"( *if self\.classifier == \"logistic\".*?)raise ValueError",
                          build, re.S).group(1).rstrip().splitlines()[:12]))

print("\n― 실제로 1 fold 돌려본다 " + "―" * 28)
rows, meta = load_label_dataset()
fold = list(make_lodo_folds(rows))[0]
fit_rows, _, test_rows = fold.split(rows)
labels = [r["primary_action"] for r in fit_rows]

spec = B.WORD_CHAR_BALANCED
pipe = B.replace(spec, class_weight=B._resolved_class_weight(spec, labels)).build()
pipe.fit(B._model_input(spec, fit_rows), labels)
pred = pipe.predict(B._model_input(spec, test_rows))
gold = [r["primary_action"] for r in test_rows]

print(f"데이터셋 {meta['dataset_version']} · 평가 문서 {fold.test_document}")
print(f"학습 {len(fit_rows)} / 평가 {len(test_rows)}건")
print(f"macro F1 {f1_score(gold, pred, labels=list(B.LABELS), average='macro', zero_division=0):.4f}")
# 파이프라인은 ['features', 'clf'] 두 단계이고, features는 word와 char를 잇는
# FeatureUnion이다. 두 어휘가 따로 쌓이므로 나눠 센다.
for name, vec in pipe.named_steps["features"].transformer_list:
    print(f"  {name:<5} 어휘 {len(vec.get_feature_names_out()):>7,}개  {vec.analyzer} {vec.ngram_range}")

In [ ]:
"""B. 인코더 파인튜닝 — 학습 루프의 핵심.

A와 결정적으로 다른 점: A는 어휘 가중치만 배우고 **문장을 못 본다**(n-gram 봉지).
B는 사전학습된 트랜스포머 전체를 갱신하므로 어순과 문맥이 들어온다.
그 대가로 GPU가 필요하다 — 이 셀은 코드만 보여주고 학습은 하지 않는다.
"""
import inspect
import textwrap

from scripts.modeling import finetune as F

source = inspect.getsource(F.train_one_fold)

print("― 모델·옵티마이저 준비 " + "―" * 30)
head, _, rest = source.partition("history: list[EpochLog]")
print(textwrap.dedent(head[head.index("labels = BINARY_LABELS"):]).rstrip())

print("\n― 학습 루프 (기울기 누적) " + "―" * 26)
loop = rest[rest.index("for epoch in range"):]
print(textwrap.dedent(loop[: loop.index("if index % args.grad_accum")]).rstrip())

print("\n― 왜 누적하는가 " + "―" * 34)
print(textwrap.fill(
    "배치를 키우면 메모리가 터지고, 줄이면 기울기가 흔들린다. 작은 배치로 여러 번 "
    "역전파하되 optimizer.step()은 몇 번에 한 번만 걸어 유효 배치를 키운다. "
    "loss를 grad_accum으로 나누는 것이 핵심 — 그래야 누적 횟수를 바꿔도 기울기 크기가 "
    "유지돼서 learning rate를 다시 잡지 않아도 된다.", 78))

print(f"\n클래스 가중치: {inspect.signature(F.class_weights)}")
print(textwrap.indent(inspect.getdoc(F.train_one_fold) or "", "  "))

In [ ]:
"""C. 앙상블 — 학습하지 않는다. D. 다중 헤드 — 아직 없다.

C는 저장된 OOF 예측을 묶기만 한다. 그래서 재실행이 싸고, 조합을 바꿔 여러 번
재볼 수 있다. 대신 **멤버가 이미 학습돼 있어야** 한다.
"""
import inspect

import torch
from torch import nn
from transformers import AutoModel

from scripts.evaluation import finetune_ensemble as E

print("― C. 다수결 " + "―" * 38)
print(inspect.getsource(E.vote).rstrip())

print("\n― C가 오르려면 두 조건이 필요하다 " + "―" * 20)
print("  1) 틀리는 자리가 다를 것   2) 멤버 각각이 충분히 강할 것")
print("  트리는 1만 만족해 실패했다 → reports/current/v5/tree_family_results.md")


class MultiHeadEncoder(nn.Module):
    """D. 공유 인코더 하나에 헤드를 여럿 단다 — 미구현 스케치.

    `blockers`는 보조 과제로 쓰지 않는다. 결정 21이
    `primary_action = f(blockers, cost_basis)`라 같은 것을 두 번 묻는 셈이다
    (WORKLOG §4에서 접은 이유). `build_difficulty`·`domain_dependency`는
    규칙에 들어가지 않으므로 인코더에 **새 지도신호**가 된다.
    """

    def __init__(self, name="klue/roberta-base", aux_sizes=(3, 3), aux_weight=0.3):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(name)
        hidden = self.encoder.config.hidden_size
        self.main_head = nn.Linear(hidden, 3)              # primary_action
        self.aux_heads = nn.ModuleList(nn.Linear(hidden, n) for n in aux_sizes)
        self.aux_weight = aux_weight                        # 보조 손실을 얼마나 섞을지

    def forward(self, input_ids, attention_mask, labels=None, aux_labels=None):
        pooled = self.encoder(input_ids=input_ids,
                              attention_mask=attention_mask).last_hidden_state[:, 0]
        logits = self.main_head(pooled)
        aux_logits = [head(pooled) for head in self.aux_heads]
        if labels is None:
            return logits, aux_logits
        loss = nn.functional.cross_entropy(logits, labels)
        # 보조 손실은 가중치를 낮춰 더한다. 1.0으로 두면 주 과제가 밀린다.
        for head_logits, target in zip(aux_logits, aux_labels):
            loss = loss + self.aux_weight * nn.functional.cross_entropy(head_logits, target)
        return loss, logits


print("\n― D. 구조 " + "―" * 40)
print(inspect.getsource(MultiHeadEncoder.__init__).rstrip())
print("\n순전파에서 손실을 어떻게 합치는가:")
print(inspect.getsource(MultiHeadEncoder.forward).rstrip())
print("\n※ 정의만 했고 학습하지 않는다. 가중치를 받으려면 GPU가 필요하다.")

## 읽는 법

**1. fold 평균과 통합 OOF를 섞지 않는다.** 같은 `word+char`가 v4에서 fold 평균
0.614, 통합 OOF 0.638이다. 문서마다 크기가 달라(49~192건) 통합하면 큰 문서가
평균을 끌어간다. 비교는 **fold 평균**으로 한다.

**2. 파인튜닝 단독은 기준선과 거의 같다.** `ftL42` 0.617 대 `word+char` 0.614다.
GPU를 쓰고도 +0.003이다. **이득은 섞을 때 나온다** — 앙상블 0.643(+0.029).
파인튜닝의 값어치는 "더 센 모델"이 아니라 "**다르게 틀리는 모델**"에 있다.

**3. 경계는 어느 방식으로도 안 줄어든다.** v4에서 앙상블이 오답을 294 → 255건으로
39건 줄이는 동안 경계 혼동은 98 → 97건, 한 건 줄었다. 오른 몫은 전부 경계 밖이다
(통상수용 F1 0.795→0.838, 견적반영 0.545→0.590). v5에서도 경계는 127건(오답의 32%)
그대로다.

**4. D는 아직 숫자가 없다.** 위 코드는 구조 스케치이지 측정이 아니다. 기대할 근거는
"보조 축이 결정 21 규칙 밖에 있어 새 정보"라는 것뿐이고, 반대 근거로는 유형을
**입력**에 넣었을 때 0.589로 떨어진 기록이 있다(가이드 02 #5). 다만 입력 결합과
보조 손실은 메커니즘이 다르므로 그 실패가 이쪽 실패를 뜻하지는 않는다.

## 다음

v5로 B를 재학습해야 C를 다시 잴 수 있다. 현재 `finetune_results.json`은 전부 v4
라벨 기준이다. 명령은 `RFP_DATASET_VERSION=v5`만 주면 된다.

```
RFP_DATASET_VERSION=v5 python -m scripts.modeling.finetune --model klue/roberta-large --seed 42
RFP_DATASET_VERSION=v5 python -m scripts.evaluation.finetune_ensemble
```

관련 노트북: `15_finetuning.ipynb`(파인튜닝과 앙상블의 겹침),
`12_candidate_ensemble.ipynb`(TF-IDF 계열 조합), `18_decision_structure.ipynb`
(OvR·OvO·캐스케이드).